In [ ]:
import socket
import pickle

soc = socket.socket()
hostname = "localhost"  # 127.0.0.1 #0.0.0.0
port = 5000
soc.bind((hostname, port))
soc.listen(5)
conn, addr = soc.accept()
print("Device Connected")

while True:
    #     msg = b"Hello World"
    msg = pickle.dumps(input("Enter the message : "))
    conn.send(msg)
    if msg == pickle.dumps("exit"):
        break

In [ ]:
import socket

mySocket = socket.socket()
mySocket.bind(("localhost", 5000))
mySocket.listen(5)

print("Waiting for C# game...")
conn, addr = mySocket.accept()
print("Device Connected")

msg = bytes("Hello from Python", "utf-8")
conn.send(msg)

conn.close()
mySocket.close()

Waiting for C# game...


In [1]:
import asyncio
import csv
import socket
from bleak import BleakScanner

def load_users():
    users = []
    with open("users.csv", "r", encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            name = row[0].strip()
            bt_id = row[1].strip().lower()
            users.append((name, bt_id))
    return users

def send_login_to_csharp(user_name):
    soc = socket.socket()
    soc.bind(("localhost", 5000))
    soc.listen(1)

    print("Waiting for C# game to connect...")
    conn, addr = soc.accept()
    print("C# connected:", addr)

    msg = "LOGIN:" + user_name
    conn.send(msg.encode("utf-8"))
    print("Sent:", msg)

    conn.close()
    soc.close()

async def scan_bluetooth_devices():
    users = load_users()
    devices = await BleakScanner.discover()

    for device in devices:
        device_address = device.address.strip().lower()

        for user in users:
            if device_address == user[1]:
                print("MATCH FOUND!")
                print("User Logged In:", user[0])
                send_login_to_csharp(user[0])
                return

    print("No matching user found.")

async def main():
    await scan_bluetooth_devices()

await main()

ModuleNotFoundError: No module named 'bleak'

In [ ]:
import asyncio
import socket
import cv2
import mediapipe as mp
import time
import os
import json
import numpy as np
import speech_recognition as sr
import pyttsx3

from dollarpy import Recognizer, Template, Point
from ultralytics import YOLO


# --- config ---

CAMERA_INDEX = 1
SOCKET_HOST = "localhost"
SOCKET_PORT = 5000

YOLO_CONFIDENCE = 0.50
YOLO_EVERY_N_FRAMES = 10

GESTURE_COOLDOWN = 1.0
OBJECT_COOLDOWN = 0.7

EMOTION_EVERY_N_FRAMES = 15
EMOTION_COOLDOWN = 2.0

SKELETON_ZONE_COOLDOWN = 1.5

FACE_DIR = "face_users"
FACE_MODEL_PATH = "face_model.yml"
FACE_LABELS_PATH = "face_labels.json"

FACE_SIZE = (200, 200)
UNKNOWN_SECONDS_BEFORE_REGISTER = 5
SAMPLES_NEEDED = 40
RECOGNITION_CONFIDENCE_LIMIT = 70
REQUIRED_LOGIN_HITS = 8


# --- progress (progress.json) ---
# saves which level each user left off at
# only ever moves forward, never back

PROGRESS_FILE = "progress.json"


def load_all_progress():
    if not os.path.exists(PROGRESS_FILE):
        return {}
    try:
        with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print("Progress load error:", e)
        return {}


def save_all_progress(data):
    try:
        with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4)
    except Exception as e:
        print("Progress save error:", e)


def get_user_progress(username):
    data = load_all_progress()
    if username in data:
        return data[username].get("level", 1)
    return 1


def create_user_progress(username):
    data = load_all_progress()
    if username not in data:
        data[username] = {"level": 1}
        save_all_progress(data)
        print(f"Progress created: {username} starts at level 1")
    else:
        print(f"{username} already saved at level {data[username]['level']}")


def update_user_progress(username, new_level):
    data = load_all_progress()
    current_level = data.get(username, {}).get("level", 1)
    if new_level > current_level:
        data[username] = {"level": new_level}
        save_all_progress(data)
        print(f"{username} saved at level {new_level}")
    else:
        print(f"{username} already at level {current_level}, no update needed")


def delete_user_progress(username):
    data = load_all_progress()
    if username in data:
        del data[username]
        save_all_progress(data)
        print(f"Progress reset for {username}")
    else:
        print(f"{username} not found in progress")


# --- word pools and levels (words.json) ---
# teachers edit this through the teacher panel
# levels_boy and levels_girl define the sentence order per gender

WORDS_FILE = "words.json"

WORDS_DEFAULT = {
    "subjects": ["I", "You", "He", "She"],
    "verbs":    ["eat", "read", "kick", "drink"],
    "objects":  ["apple", "book", "ball", "milk"],
    "levels_boy": [
        {"subject": "I",   "verb": "eat",   "object": "apple"},
        {"subject": "He",  "verb": "kick",  "object": "ball"},
        {"subject": "You", "verb": "read",  "object": "book"},
        {"subject": "She", "verb": "drink", "object": "milk"}
    ],
    "levels_girl": [
        {"subject": "You", "verb": "read",  "object": "book"},
        {"subject": "She", "verb": "drink", "object": "milk"},
        {"subject": "I",   "verb": "eat",   "object": "apple"},
        {"subject": "He",  "verb": "kick",  "object": "ball"}
    ]
}


def load_words():
    if not os.path.exists(WORDS_FILE):
        save_words(WORDS_DEFAULT)
        return WORDS_DEFAULT
    try:
        with open(WORDS_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print("Words load error:", e)
        return WORDS_DEFAULT


def save_words(data):
    try:
        with open(WORDS_FILE, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4)
    except Exception as e:
        print("Words save error:", e)


# make sure words.json exists before anything else runs
load_words()


def teacher_panel():
    print("\n" + "=" * 40)
    print("       TEACHER PANEL")
    print("=" * 40)

    while True:
        words = load_words()

        boy_levels = words.get("levels_boy", [])
        print("\nLevels (" + str(len(boy_levels)) + " total):")
        for i, lvl in enumerate(boy_levels):
            print(f"  {i+1}: {lvl['subject']} {lvl['verb']} {lvl['object']}")

        print("\nWord pools:")
        print("  Subjects:", words["subjects"])
        print("  Verbs:   ", words["verbs"])
        print("  Objects: ", words["objects"])

        print("\n  1. Add a word")
        print("  2. Remove a word")
        print("  3. Update a word")
        print("  4. Add a level")
        print("  5. Remove last level")
        print("  6. Exit")

        choice = input("\nOption: ").strip()

        if choice == "1":
            category = input("Category (subjects / verbs / objects): ").strip().lower()
            if category not in words:
                print("Invalid category.")
                continue
            word = input(f"Word to add: ").strip()
            if word == "":
                print("Empty, skipping.")
                continue
            if word.lower() in [w.lower() for w in words[category]]:
                print(f"'{word}' already exists.")
                continue
            words[category].append(word)
            save_words(words)
            print(f"Added '{word}'.")

        elif choice == "2":
            category = input("Category (subjects / verbs / objects): ").strip().lower()
            if category not in words:
                print("Invalid category.")
                continue
            print(f"Current:", words[category])
            word = input("Word to remove: ").strip()
            match = next((w for w in words[category] if w.lower() == word.lower()), None)
            if match is None:
                print(f"'{word}' not found.")
                continue
            if len(words[category]) <= 1:
                print("Can't remove the last word.")
                continue
            words[category].remove(match)
            save_words(words)
            print(f"Removed '{match}'.")

        elif choice == "3":
            category = input("Category (subjects / verbs / objects): ").strip().lower()
            if category not in words:
                print("Invalid category.")
                continue
            print(f"Current:", words[category])
            old_word = input("Word to replace: ").strip()
            match = next((w for w in words[category] if w.lower() == old_word.lower()), None)
            if match is None:
                print(f"'{old_word}' not found.")
                continue
            new_word = input(f"Replace with: ").strip()
            if new_word == "":
                print("Empty, skipping.")
                continue
            idx = words[category].index(match)
            words[category][idx] = new_word
            save_words(words)
            print(f"Updated '{match}' to '{new_word}'.")

        elif choice == "4":
            print("New level — added to both boy and girl.")
            subject = input("Subject: ").strip()
            verb    = input("Verb: ").strip()
            obj     = input("Object: ").strip()

            if subject == "" or verb == "" or obj == "":
                print("All three required.")
                continue

            for cat, word in [("subjects", subject), ("verbs", verb), ("objects", obj)]:
                if word.lower() not in [w.lower() for w in words[cat]]:
                    words[cat].append(word)

            new_level = {"subject": subject, "verb": verb, "object": obj}
            words.setdefault("levels_boy",  []).append(new_level)
            words.setdefault("levels_girl", []).append(new_level)
            save_words(words)
            print(f"Level added: {subject} {verb} {obj}")

        elif choice == "5":
            boy_levels  = words.get("levels_boy",  [])
            girl_levels = words.get("levels_girl", [])
            if not boy_levels:
                print("No levels to remove.")
                continue
            removed = boy_levels[-1]
            words["levels_boy"]  = boy_levels[:-1]
            words["levels_girl"] = girl_levels[:-1] if girl_levels else []
            save_words(words)
            print(f"Removed: {removed['subject']} {removed['verb']} {removed['object']}")

        elif choice == "6":
            print("Done.")
            break

        else:
            print("Invalid option.")


# --- mediapipe ---

mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic


# --- yolo ---

yolo_model = YOLO("yolov8n.pt")


# --- socket ---

def send_message(conn, msg):
    try:
        conn.send((msg + "\n").encode("utf-8"))
        print("Sent:", msg)
    except Exception as e:
        print("Send Error:", str(e))


def get_level_answer(mode, level_index):
    words = load_words()
    key = "levels_boy" if mode == "m" else "levels_girl"
    levels = words.get(key, [])
    if level_index < 0 or level_index >= len(levels):
        return None
    lvl = levels[level_index]
    return f"{lvl['subject']} {lvl['verb']} {lvl['object']}"


def get_total_levels(mode):
    words = load_words()
    key = "levels_boy" if mode == "m" else "levels_girl"
    return len(words.get(key, []))


def send_level_info(conn, mode, level):
    words = load_words()
    key = "levels_boy" if mode == "m" else "levels_girl"
    levels = words.get(key, [])
    total = len(levels)
    level_index = level - 1

    send_message(conn, f"TOTALLEVELS:{total}")
    time.sleep(0.05)

    if 0 <= level_index < total:
        lvl = levels[level_index]
        answer = f"{lvl['subject']} {lvl['verb']} {lvl['object']}"
        send_message(conn, f"LEVELANSWER:{answer}")
    else:
        print(f"Invalid level index: {level_index}")


def send_word_pools(conn):
    words = load_words()
    time.sleep(0.1)
    send_message(conn, "WORDS:subjects:" + ",".join(words["subjects"]))
    time.sleep(0.05)
    send_message(conn, "WORDS:verbs:" + ",".join(words["verbs"]))
    time.sleep(0.05)
    send_message(conn, "WORDS:objects:" + ",".join(words["objects"]))


# --- voice ---

def speak(text):
    try:
        engine = pyttsx3.init()
        engine.say(text)
        engine.runAndWait()
    except Exception as e:
        print("Speak error:", str(e))


def listen_once(prompt_text, timeout=7, phrase_time_limit=4):
    recognizer = sr.Recognizer()
    speak(prompt_text)
    print(prompt_text)
    try:
        with sr.Microphone() as source:
            print("Listening...")
            recognizer.adjust_for_ambient_noise(source, duration=0.8)
            audio = recognizer.listen(source, timeout=timeout, phrase_time_limit=phrase_time_limit)
        text = recognizer.recognize_google(audio)
        text = text.strip()
        print("Heard:", text)
        return text
    except sr.WaitTimeoutError:
        print("Nothing heard.")
        return ""
    except sr.UnknownValueError:
        print("Could not understand.")
        return ""
    except Exception as e:
        print("Voice error:", str(e))
        return ""


def get_user_name_by_voice():
    for attempt in range(3):
        name = listen_once("Please say your name", timeout=7, phrase_time_limit=4)
        if name != "":
            name = name.strip().replace(" ", "_")
            name = "".join(ch for ch in name if ch.isalnum() or ch == "_")
            if name != "":
                speak("Your name is " + name)
                return name
        speak("Didn't catch that. Try again.")
    return "New_User"


def get_user_mode_by_voice():
    for attempt in range(3):
        mode_text = listen_once("Say boy mode or girl mode", timeout=6, phrase_time_limit=3)
        mode_text = mode_text.lower().strip()
        if "boy" in mode_text or "male" in mode_text or mode_text == "m":
            speak("Boy mode selected")
            return "m"
        if "girl" in mode_text or "female" in mode_text or mode_text == "f":
            speak("Girl mode selected")
            return "f"
        speak("Try again. Boy mode or girl mode.")
    return "m"


# --- face login ---

def get_face_detector():
    return cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")


def face_model_exists():
    return os.path.exists(FACE_MODEL_PATH) and os.path.exists(FACE_LABELS_PATH)


def load_face_model():
    recognizer = cv2.face.LBPHFaceRecognizer_create()
    recognizer.read(FACE_MODEL_PATH)
    with open(FACE_LABELS_PATH, "r", encoding="utf-8") as f:
        label_data = json.load(f)
    return recognizer, label_data


def train_face_model():
    os.makedirs(FACE_DIR, exist_ok=True)
    recognizer = cv2.face.LBPHFaceRecognizer_create()
    faces = []
    labels = []
    label_data = {}
    label_id = 0

    for folder_name in os.listdir(FACE_DIR):
        folder_path = os.path.join(FACE_DIR, folder_name)
        if not os.path.isdir(folder_path):
            continue
        parts = folder_name.rsplit("_", 1)
        if len(parts) != 2:
            continue
        name = parts[0]
        mode = parts[1]
        label_data[label_id] = {"name": name, "mode": mode}
        for img_name in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, FACE_SIZE)
            faces.append(img)
            labels.append(label_id)
        label_id += 1

    if len(faces) == 0:
        print("No face images found.")
        return False

    recognizer.train(faces, np.array(labels))
    recognizer.save(FACE_MODEL_PATH)
    with open(FACE_LABELS_PATH, "w", encoding="utf-8") as f:
        json.dump(label_data, f, indent=4)

    print("Model trained. Users:", label_data)
    return True


def register_new_face_user(name, mode):
    mode = mode.lower().strip()
    if mode not in ["m", "f"]:
        mode = "m"

    safe_name = name.strip().replace(" ", "_")
    safe_name = "".join(ch for ch in safe_name if ch.isalnum() or ch == "_")
    save_dir = os.path.join(FACE_DIR, safe_name + "_" + mode)
    os.makedirs(save_dir, exist_ok=True)

    face_cascade = get_face_detector()
    cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
    if not cap.isOpened():
        print("Camera could not open.")
        return False

    count = 0
    print("Registering:", safe_name, mode)
    speak("Look at the camera.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=5, minSize=(80, 80))

        for (x, y, w, h) in faces:
            face_img = gray[y:y+h, x:x+w]
            face_img = cv2.resize(face_img, FACE_SIZE)
            cv2.imwrite(os.path.join(save_dir, str(count) + ".jpg"), face_img)
            count += 1
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
            cv2.putText(frame, f"Saving {count}/{SAMPLES_NEEDED}",
                        (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            time.sleep(0.08)
            break

        cv2.imshow("Register New Face", frame)

        if count >= SAMPLES_NEEDED:
            speak("Done.")
            break
        if cv2.waitKey(1) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()

    if count >= 10:
        train_face_model()
        return True

    print("Not enough samples.")
    return False


def auto_face_login_or_register(conn):
    face_cascade = get_face_detector()
    cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
    if not cap.isOpened():
        print("Camera could not open.")
        return False, None, None

    recognizer = None
    label_data = None

    if face_model_exists():
        recognizer, label_data = load_face_model()
        speak("Face login started.")
    else:
        speak("No users registered. Look at the camera.")

    unknown_start_time = None
    last_name = ""
    hit_count = 0

    print("Face login — ESC to skip.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=5, minSize=(80, 80))

        face_found = False
        recognized = False

        for (x, y, w, h) in faces:
            face_found = True
            face_img = cv2.resize(gray[y:y+h, x:x+w], FACE_SIZE)

            if recognizer is not None:
                label_id, confidence = recognizer.predict(face_img)
                label_id_str = str(label_id)

                if label_id_str in label_data and confidence < RECOGNITION_CONFIDENCE_LIMIT:
                    name = label_data[label_id_str]["name"]
                    mode = label_data[label_id_str].get("mode", label_data[label_id_str].get("gender", "m"))

                    recognized = True
                    unknown_start_time = None

                    if name == last_name:
                        hit_count += 1
                    else:
                        last_name = name
                        hit_count = 1

                    cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
                    cv2.putText(frame, f"{name} {hit_count}/{REQUIRED_LOGIN_HITS}",
                                (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

                    if hit_count >= REQUIRED_LOGIN_HITS:
                        create_user_progress(name)
                        level = get_user_progress(name)

                        send_message(conn, f"LOGIN:{name}:{mode}:{level}")
                        send_word_pools(conn)
                        send_level_info(conn, mode, level)

                        cap.release()
                        cv2.destroyAllWindows()
                        speak("Welcome " + name)
                        print(f"Logged in: {name} mode={mode} level={level}")
                        return True, name, mode

                else:
                    cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 255), 2)
                    cv2.putText(frame, "Unknown", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

            else:
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 255), 2)
                cv2.putText(frame, "New face", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

            break

        if face_found and not recognized:
            if unknown_start_time is None:
                unknown_start_time = time.time()
                speak("Unknown face. Hold still.")

            elapsed = time.time() - unknown_start_time
            cv2.putText(frame, f"Unknown: {round(elapsed, 1)}s",
                        (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

            if elapsed >= UNKNOWN_SECONDS_BEFORE_REGISTER:
                cap.release()
                cv2.destroyAllWindows()

                new_name = get_user_name_by_voice()
                new_mode = get_user_mode_by_voice()

                registered = register_new_face_user(new_name, new_mode)

                if registered:
                    create_user_progress(new_name)
                    level = get_user_progress(new_name)
                    send_message(conn, f"LOGIN:{new_name}:{new_mode}:{level}")
                    send_word_pools(conn)
                    send_level_info(conn, new_mode, 1)
                    speak("Welcome " + new_name)
                    return True, new_name, new_mode

                return False, None, None

        elif not face_found:
            unknown_start_time = None
            hit_count = 0

        cv2.imshow("Face Login", frame)

        if cv2.waitKey(1) & 0xFF == 27:
            print("Login skipped.")
            break

    cap.release()
    cv2.destroyAllWindows()
    return False, None, None


# --- gesture templates ($1 recognizer) ---
# extracts the right index fingertip trail from a video
# used to build gesture templates for dollarpy

def getPoints(videoURL, label):
    cap = cv2.VideoCapture(videoURL)
    points = []
    stroke_id = 1

    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            results = holistic.process(image)
            image.flags.writeable = True

            if results.right_hand_landmarks:
                tip = results.right_hand_landmarks.landmark[8]
                h, w, _ = frame.shape
                points.append(Point(int(tip.x * w), int(tip.y * h), stroke_id))

    cap.release()
    cv2.destroyAllWindows()
    return points


templates = []


def add_template(label, video_path):
    points = getPoints(video_path, label)
    print(label, video_path, "points =", len(points))
    if len(points) > 0:
        templates.append(Template(label, points))
    else:
        print("Skipped (no points):", video_path)


add_template("circle",      "Dataset/Dataset/circle/circle_1.mp4")
add_template("circle",      "Dataset/Dataset/circle/circle_2.mp4")
add_template("slide_left",  "Dataset/Dataset/slide_left/slide_left_1.mp4")
add_template("slide_left",  "Dataset/Dataset/slide_left/slide_left_2.mp4")
add_template("slide_right", "Dataset/Dataset/slide_right/slide_right_1.mp4")
add_template("slide_right", "Dataset/Dataset/slide_right/slide_right_2.mp4")
add_template("fist",        "Dataset/Dataset/fist/fist_1.mp4")
add_template("fist",        "Dataset/Dataset/fist/fist_2.mp4")
add_template("peace",       "Dataset/Dataset/peace/peace_1.mp4")
add_template("peace",       "Dataset/Dataset/peace/peace_2.mp4")

print("Templates loaded:", len(templates))

if len(templates) == 0:
    raise Exception("No templates loaded.")

dollar_recognizer = Recognizer(templates)


# --- gesture sending ---

last_label = ""
last_time = 0


def send_gesture(conn, label):
    global last_label, last_time
    now = time.time()

    if label == last_label and now - last_time < GESTURE_COOLDOWN:
        return

    mapping = {
        "slide_right": "GESTURE:RIGHT",
        "slide_left":  "GESTURE:LEFT",
        "circle":      "GESTURE:CIRCLE",
        "fist":        "GESTURE:FIST",
        "peace":       "GESTURE:PEACE",
    }

    if label not in mapping:
        return

    send_message(conn, mapping[label])
    last_label = label
    last_time = now


# --- skeleton zone detection ---
# maps the fingertip position to one of three word slots on screen
# only fires in the lower half of the camera frame to avoid conflicts
# with gestures made at the top
#
# box positions measured from the actual game screenshot (1456px wide):
#   subject: 130-390, verb: 610-860, object: 1080-1340
# these get scaled down to camera coords using the window width

last_zone = ""
last_zone_time = 0

GAME_WINDOW_WIDTH = 1920
IS_LASER_MODE = False


def get_skeleton_zone(x, y, frame_width, frame_height):
    if y < frame_height // 2:
        return None

    scale = GAME_WINDOW_WIDTH / frame_width

    if int(130 / scale) <= x <= int(390 / scale):
        return "SUBJECT"
    elif int(610 / scale) <= x <= int(860 / scale):
        return "VERB"
    elif int(1080 / scale) <= x <= int(1340 / scale):
        return "OBJECT"
    return None


def send_skeleton_zone(conn, zone):
    global last_zone, last_zone_time
    now = time.time()
    if zone == last_zone and now - last_zone_time < SKELETON_ZONE_COOLDOWN:
        return
    send_message(conn, "ZONE:" + zone)
    last_zone = zone
    last_zone_time = now


# --- yolo object detection ---
# ball, book, apple each send a distinct command to C#
# separate from hand gestures

last_object = ""
last_object_time = 0


def send_detected_object(conn, object_name):
    global last_object, last_object_time
    now = time.time()

    if object_name == last_object and now - last_object_time < OBJECT_COOLDOWN:
        return

    mapping = {
        "sports ball": "OBJECT:BALL",
        "book":        "OBJECT:BOOK",
        "apple":       "OBJECT:APPLE",
    }

    if object_name not in mapping:
        return

    send_message(conn, mapping[object_name])
    last_object = object_name
    last_object_time = now


def process_yolo_detection(conn, frame):
    yolo_results = yolo_model(frame, verbose=False)
    for result in yolo_results:
        for box in result.boxes:
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
            object_name = yolo_model.names[class_id]
            if confidence >= YOLO_CONFIDENCE and object_name in ["sports ball", "book", "apple"]:
                print("YOLO:", object_name, round(confidence, 2))
                send_detected_object(conn, object_name)


# --- emotion detection ---
# uses mediapipe face landmarks to estimate expression
# ratios based on mouth width, openness, and corner drop

last_emotion = ""
last_emotion_time = 0


def send_emotion(conn, emotion_name):
    global last_emotion, last_emotion_time
    now = time.time()
    emotion_name = emotion_name.upper().strip()

    if emotion_name == last_emotion and now - last_emotion_time < EMOTION_COOLDOWN:
        return
    if emotion_name not in ["HAPPY", "NEUTRAL", "SAD", "SURPRISE"]:
        return

    send_message(conn, "EMOTION:" + emotion_name)
    last_emotion = emotion_name
    last_emotion_time = now


def classify_expression_from_face_landmarks(face_landmarks):
    if face_landmarks is None:
        return "NEUTRAL"

    lm = face_landmarks.landmark
    mouth_width  = abs(lm[291].x - lm[61].x)
    mouth_open   = abs(lm[14].y  - lm[13].y)
    face_height  = abs(lm[152].y - lm[1].y)

    if face_height == 0:
        return "NEUTRAL"

    open_ratio  = mouth_open / face_height
    smile_ratio = mouth_width / face_height
    sad_ratio   = ((lm[61].y + lm[291].y) / 2) - ((lm[37].y + lm[267].y) / 2)

    print("mouth_open:", round(open_ratio, 2), "smile:", round(smile_ratio, 2), "sad:", round(sad_ratio, 3))

    if open_ratio  > 0.22: return "SURPRISE"
    if smile_ratio > 0.95: return "HAPPY"
    if sad_ratio   > 0.035: return "SAD"
    return "NEUTRAL"


# --- main camera loop ---

def recognize_live_gesture(conn, current_user, current_mode="m"):
    cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
    if not cap.isOpened():
        print("Camera could not open.")
        return

    conn.setblocking(False)

    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        points = []
        stroke_id = 1
        tracking = False
        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            h, w, _ = frame.shape

            try:
                data = conn.recv(1024)
                if data:
                    incoming = data.decode("utf-8").strip()
                    print("From C#:", incoming)

                    if incoming.startswith("WINDOWSIZE:"):
                        try:
                            global GAME_WINDOW_WIDTH
                            GAME_WINDOW_WIDTH = int(incoming.split(":")[1])
                        except Exception as e:
                            print("WINDOWSIZE error:", e)

                    elif incoming.startswith("MODE:"):
                        global IS_LASER_MODE
                        IS_LASER_MODE = (incoming.split(":")[1].strip() == "LASER")
                        print("Mode:", "LASER" if IS_LASER_MODE else "GESTURE")

                    elif incoming.startswith("LEVELUP:") and current_user:
                        try:
                            new_level = int(incoming.split(":")[1])
                            update_user_progress(current_user, new_level)
                            speak("Level " + str(new_level) + " unlocked!")
                            send_level_info(conn, current_mode, new_level)
                        except Exception as e:
                            print("LEVELUP error:", e)

                    elif incoming == "WIN" and current_user:
                        speak("Congratulations! You win!")
                        print("All levels complete:", current_user)

            except BlockingIOError:
                pass
            except Exception as e:
                print("Receive error:", e)

            if frame_count % YOLO_EVERY_N_FRAMES == 0:
                process_yolo_detection(conn, frame)

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            results = holistic.process(image)
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            if frame_count % EMOTION_EVERY_N_FRAMES == 0:
                emotion = classify_expression_from_face_landmarks(results.face_landmarks)
                if emotion != "NEUTRAL":
                    print(f"[EMOTION] {emotion} frame {frame_count}")
                send_emotion(conn, emotion)

            if results.right_hand_landmarks:
                mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

                tip = results.right_hand_landmarks.landmark[8]
                x = int(tip.x * w)
                y = int(tip.y * h)

                points.append(Point(x, y, stroke_id))
                cv2.circle(image, (x, y), 8, (0, 255, 0), -1)
                tracking = True

                zone = get_skeleton_zone(x, y, w, h)
                if zone:
                    send_skeleton_zone(conn, zone)

                if IS_LASER_MODE:
                    send_message(conn, f"FINGER:{x},{y}")

            else:
                if tracking and len(points) > 10:
                    try:
                        result = dollar_recognizer.recognize(points)
                        if result is not None:
                            print("Predicted:", result[0], "Score:", result[1])
                            send_gesture(conn, result[0])
                    except Exception as e:
                        print("Recognition error:", str(e))

                points = []
                tracking = False

            cv2.imshow("HCI Game — Camera Feed", image)

            if cv2.waitKey(1) & 0xFF == 27:
                break

    cap.release()
    cv2.destroyAllWindows()


# --- entry point ---

async def main():
    # change to "2" to open the teacher panel instead
    choice = "1"

    if choice == "2":
        teacher_panel()
        print("Done. Change choice back to 1 to start the game.")
        return

    soc = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    soc.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    soc.bind((SOCKET_HOST, SOCKET_PORT))
    soc.listen(1)

    print("Waiting for C# to connect on port 5000...")

    conn, addr = soc.accept()
    print("C# connected:", addr)

    logged_in, current_user, current_mode = auto_face_login_or_register(conn)

    if not logged_in:
        send_message(conn, "LOGIN:Guest:m:1")
        send_word_pools(conn)
        send_level_info(conn, "m", 1)
        current_user = "Guest"
        current_mode = "m"

    recognize_live_gesture(conn, current_user, current_mode)

    conn.close()
    soc.close()


await main()

circle Dataset/Dataset/circle/circle_1.mp4 points = 69
circle Dataset/Dataset/circle/circle_2.mp4 points = 105
slide_left Dataset/Dataset/slide_left/slide_left_1.mp4 points = 40
slide_left Dataset/Dataset/slide_left/slide_left_2.mp4 points = 26
slide_right Dataset/Dataset/slide_right/slide_right_1.mp4 points = 12
slide_right Dataset/Dataset/slide_right/slide_right_2.mp4 points = 15
fist Dataset/Dataset/fist/fist_1.mp4 points = 59
fist Dataset/Dataset/fist/fist_2.mp4 points = 51
peace Dataset/Dataset/peace/peace_1.mp4 points = 64
peace Dataset/Dataset/peace/peace_2.mp4 points = 77
Templates loaded: 10
Waiting for C# to connect on port 5000...
C# connected: ('127.0.0.1', 62897)


In [1]:
import cv2
cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)
ret, frame = cap.read()
if ret:
    print("Camera resolution:", frame.shape[1], "x", frame.shape[0])
cap.release()

Camera resolution: 640 x 480


In [2]:
import json, os

WORDS_FILE = "words.json"
WORDS_DEFAULT = {
    "subjects": ["I", "You", "He", "She"],
    "verbs":    ["eat", "read", "kick", "drink"],
    "objects":  ["apple", "book", "ball", "milk"],
    "levels_boy": [
        {"subject": "I",   "verb": "eat",   "object": "apple"},
        {"subject": "He",  "verb": "kick",  "object": "ball"},
        {"subject": "You", "verb": "read",  "object": "book"},
        {"subject": "She", "verb": "drink", "object": "milk"}
    ],
    "levels_girl": [
        {"subject": "You", "verb": "read",  "object": "book"},
        {"subject": "She", "verb": "drink", "object": "milk"},
        {"subject": "I",   "verb": "eat",   "object": "apple"},
        {"subject": "He",  "verb": "kick",  "object": "ball"}
    ]
}

with open(WORDS_FILE, "w", encoding="utf-8") as f:
    json.dump(WORDS_DEFAULT, f, indent=4)

print("Created at:", os.path.abspath(WORDS_FILE))

Created at: c:\Uni\hci project\hci project\HCI bullshit\words.json


In [16]:
pip --user

Note: you may need to restart the kernel to use updated packages.



Usage:   
  c:\Users\yosef\AppData\Local\Programs\Python\Python310\python.exe -m pip <command> [options]

no such option: --user


In [ ]:
import asyncio
import socket
import cv2
import mediapipe as mp
import time
import os
import json
import numpy as np
import speech_recognition as sr
import pyttsx3

from dollarpy import Recognizer, Template, Point
from ultralytics import YOLO


# --- config ---

CAMERA_INDEX = 1
SOCKET_HOST = "localhost"
SOCKET_PORT = 5000

YOLO_CONFIDENCE = 0.50
YOLO_EVERY_N_FRAMES = 10

GESTURE_COOLDOWN = 1.0
OBJECT_COOLDOWN = 0.7

EMOTION_EVERY_N_FRAMES = 15
EMOTION_COOLDOWN = 2.0

SKELETON_ZONE_COOLDOWN = 1.5

FACE_DIR = "face_users"
FACE_MODEL_PATH = "face_model.yml"
FACE_LABELS_PATH = "face_labels.json"

FACE_SIZE = (200, 200)
UNKNOWN_SECONDS_BEFORE_REGISTER = 5
SAMPLES_NEEDED = 40
RECOGNITION_CONFIDENCE_LIMIT = 70
REQUIRED_LOGIN_HITS = 8


# --- progress (progress.json) ---
# saves which level each user left off at
# only ever moves forward, never back

PROGRESS_FILE = "progress.json"


def load_all_progress():
    if not os.path.exists(PROGRESS_FILE):
        return {}
    try:
        with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print("Progress load error:", e)
        return {}


def save_all_progress(data):
    try:
        with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4)
    except Exception as e:
        print("Progress save error:", e)


def get_user_progress(username):
    data = load_all_progress()
    if username in data:
        return data[username].get("level", 1)
    return 1


def create_user_progress(username):
    data = load_all_progress()
    if username not in data:
        data[username] = {"level": 1}
        save_all_progress(data)
        print(f"Progress created: {username} starts at level 1")
    else:
        print(f"{username} already saved at level {data[username]['level']}")


def update_user_progress(username, new_level):
    data = load_all_progress()
    current_level = data.get(username, {}).get("level", 1)
    if new_level > current_level:
        data[username] = {"level": new_level}
        save_all_progress(data)
        print(f"{username} saved at level {new_level}")
    else:
        print(f"{username} already at level {current_level}, no update needed")


def delete_user_progress(username):
    data = load_all_progress()
    if username in data:
        del data[username]
        save_all_progress(data)
        print(f"Progress reset for {username}")
    else:
        print(f"{username} not found in progress")


# --- word pools and levels (words.json) ---
# teachers edit this through the teacher panel
# levels_boy and levels_girl define the sentence order per gender

WORDS_FILE = "words.json"

WORDS_DEFAULT = {
    "subjects": ["I", "You", "He", "She"],
    "verbs":    ["eat", "read", "kick", "drink"],
    "objects":  ["apple", "book", "ball", "milk"],
    "levels_boy": [
        {"subject": "I",   "verb": "eat",   "object": "apple"},
        {"subject": "He",  "verb": "kick",  "object": "ball"},
        {"subject": "You", "verb": "read",  "object": "book"},
        {"subject": "She", "verb": "drink", "object": "milk"}
    ],
    "levels_girl": [
        {"subject": "You", "verb": "read",  "object": "book"},
        {"subject": "She", "verb": "drink", "object": "milk"},
        {"subject": "I",   "verb": "eat",   "object": "apple"},
        {"subject": "He",  "verb": "kick",  "object": "ball"}
    ]
}


def load_words():
    if not os.path.exists(WORDS_FILE):
        save_words(WORDS_DEFAULT)
        return WORDS_DEFAULT
    try:
        with open(WORDS_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print("Words load error:", e)
        return WORDS_DEFAULT


def save_words(data):
    try:
        with open(WORDS_FILE, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4)
    except Exception as e:
        print("Words save error:", e)


# make sure words.json exists before anything else runs
load_words()


def teacher_panel():
    print("\n" + "=" * 40)
    print("       TEACHER PANEL")
    print("=" * 40)

    while True:
        words = load_words()

        boy_levels = words.get("levels_boy", [])
        print("\nLevels (" + str(len(boy_levels)) + " total):")
        for i, lvl in enumerate(boy_levels):
            print(f"  {i+1}: {lvl['subject']} {lvl['verb']} {lvl['object']}")

        print("\nWord pools:")
        print("  Subjects:", words["subjects"])
        print("  Verbs:   ", words["verbs"])
        print("  Objects: ", words["objects"])

        print("\n  1. Add a word")
        print("  2. Remove a word")
        print("  3. Update a word")
        print("  4. Add a level")
        print("  5. Remove last level")
        print("  6. Exit")

        choice = input("\nOption: ").strip()

        if choice == "1":
            category = input("Category (subjects / verbs / objects): ").strip().lower()
            if category not in words:
                print("Invalid category.")
                continue
            word = input(f"Word to add: ").strip()
            if word == "":
                print("Empty, skipping.")
                continue
            if word.lower() in [w.lower() for w in words[category]]:
                print(f"'{word}' already exists.")
                continue
            words[category].append(word)
            save_words(words)
            print(f"Added '{word}'.")

        elif choice == "2":
            category = input("Category (subjects / verbs / objects): ").strip().lower()
            if category not in words:
                print("Invalid category.")
                continue
            print(f"Current:", words[category])
            word = input("Word to remove: ").strip()
            match = next((w for w in words[category] if w.lower() == word.lower()), None)
            if match is None:
                print(f"'{word}' not found.")
                continue
            if len(words[category]) <= 1:
                print("Can't remove the last word.")
                continue
            words[category].remove(match)
            save_words(words)
            print(f"Removed '{match}'.")

        elif choice == "3":
            category = input("Category (subjects / verbs / objects): ").strip().lower()
            if category not in words:
                print("Invalid category.")
                continue
            print(f"Current:", words[category])
            old_word = input("Word to replace: ").strip()
            match = next((w for w in words[category] if w.lower() == old_word.lower()), None)
            if match is None:
                print(f"'{old_word}' not found.")
                continue
            new_word = input(f"Replace with: ").strip()
            if new_word == "":
                print("Empty, skipping.")
                continue
            idx = words[category].index(match)
            words[category][idx] = new_word
            save_words(words)
            print(f"Updated '{match}' to '{new_word}'.")

        elif choice == "4":
            print("New level — added to both boy and girl.")
            subject = input("Subject: ").strip()
            verb    = input("Verb: ").strip()
            obj     = input("Object: ").strip()

            if subject == "" or verb == "" or obj == "":
                print("All three required.")
                continue

            for cat, word in [("subjects", subject), ("verbs", verb), ("objects", obj)]:
                if word.lower() not in [w.lower() for w in words[cat]]:
                    words[cat].append(word)

            new_level = {"subject": subject, "verb": verb, "object": obj}
            words.setdefault("levels_boy",  []).append(new_level)
            words.setdefault("levels_girl", []).append(new_level)
            save_words(words)
            print(f"Level added: {subject} {verb} {obj}")

        elif choice == "5":
            boy_levels  = words.get("levels_boy",  [])
            girl_levels = words.get("levels_girl", [])
            if not boy_levels:
                print("No levels to remove.")
                continue
            removed = boy_levels[-1]
            words["levels_boy"]  = boy_levels[:-1]
            words["levels_girl"] = girl_levels[:-1] if girl_levels else []
            save_words(words)
            print(f"Removed: {removed['subject']} {removed['verb']} {removed['object']}")

        elif choice == "6":
            print("Done.")
            break

        else:
            print("Invalid option.")


# --- mediapipe ---

mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic


# --- yolo ---

yolo_model = YOLO("yolov8n.pt")


# --- socket ---

def send_message(conn, msg):
    try:
        conn.send((msg + "\n").encode("utf-8"))
        print("Sent:", msg)
    except Exception as e:
        print("Send Error:", str(e))


def get_level_answer(mode, level_index):
    words = load_words()
    key = "levels_boy" if mode == "m" else "levels_girl"
    levels = words.get(key, [])
    if level_index < 0 or level_index >= len(levels):
        return None
    lvl = levels[level_index]
    return f"{lvl['subject']} {lvl['verb']} {lvl['object']}"


def get_total_levels(mode):
    words = load_words()
    key = "levels_boy" if mode == "m" else "levels_girl"
    return len(words.get(key, []))


def send_level_info(conn, mode, level):
    words = load_words()
    key = "levels_boy" if mode == "m" else "levels_girl"
    levels = words.get(key, [])
    total = len(levels)
    level_index = level - 1

    send_message(conn, f"TOTALLEVELS:{total}")
    time.sleep(0.05)

    if 0 <= level_index < total:
        lvl = levels[level_index]
        answer = f"{lvl['subject']} {lvl['verb']} {lvl['object']}"
        send_message(conn, f"LEVELANSWER:{answer}")
    else:
        print(f"Invalid level index: {level_index}")


def send_word_pools(conn):
    words = load_words()
    time.sleep(0.1)
    send_message(conn, "WORDS:subjects:" + ",".join(words["subjects"]))
    time.sleep(0.05)
    send_message(conn, "WORDS:verbs:" + ",".join(words["verbs"]))
    time.sleep(0.05)
    send_message(conn, "WORDS:objects:" + ",".join(words["objects"]))


# --- voice ---

def speak(text):
    try:
        engine = pyttsx3.init()
        engine.say(text)
        engine.runAndWait()
    except Exception as e:
        print("Speak error:", str(e))


def listen_once(prompt_text, timeout=7, phrase_time_limit=4):
    recognizer = sr.Recognizer()
    speak(prompt_text)
    print(prompt_text)
    try:
        with sr.Microphone() as source:
            print("Listening...")
            recognizer.adjust_for_ambient_noise(source, duration=0.8)
            audio = recognizer.listen(source, timeout=timeout, phrase_time_limit=phrase_time_limit)
        text = recognizer.recognize_google(audio)
        text = text.strip()
        print("Heard:", text)
        return text
    except sr.WaitTimeoutError:
        print("Nothing heard.")
        return ""
    except sr.UnknownValueError:
        print("Could not understand.")
        return ""
    except Exception as e:
        print("Voice error:", str(e))
        return ""


def get_user_name_by_voice():
    for attempt in range(3):
        name = listen_once("Please say your name", timeout=7, phrase_time_limit=4)
        if name != "":
            name = name.strip().replace(" ", "_")
            name = "".join(ch for ch in name if ch.isalnum() or ch == "_")
            if name != "":
                speak("Your name is " + name)
                return name
        speak("Didn't catch that. Try again.")
    return "New_User"


def get_user_mode_by_voice():
    for attempt in range(3):
        mode_text = listen_once("Say boy mode or girl mode", timeout=6, phrase_time_limit=3)
        mode_text = mode_text.lower().strip()
        if "boy" in mode_text or "male" in mode_text or mode_text == "m":
            speak("Boy mode selected")
            return "m"
        if "girl" in mode_text or "female" in mode_text or mode_text == "f":
            speak("Girl mode selected")
            return "f"
        speak("Try again. Boy mode or girl mode.")
    return "m"


# --- face login ---

def get_face_detector():
    return cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")


def face_model_exists():
    return os.path.exists(FACE_MODEL_PATH) and os.path.exists(FACE_LABELS_PATH)


def load_face_model():
    recognizer = cv2.face.LBPHFaceRecognizer_create()
    recognizer.read(FACE_MODEL_PATH)
    with open(FACE_LABELS_PATH, "r", encoding="utf-8") as f:
        label_data = json.load(f)
    return recognizer, label_data


def train_face_model():
    os.makedirs(FACE_DIR, exist_ok=True)
    recognizer = cv2.face.LBPHFaceRecognizer_create()
    faces = []
    labels = []
    label_data = {}
    label_id = 0

    for folder_name in os.listdir(FACE_DIR):
        folder_path = os.path.join(FACE_DIR, folder_name)
        if not os.path.isdir(folder_path):
            continue
        parts = folder_name.rsplit("_", 1)
        if len(parts) != 2:
            continue
        name = parts[0]
        mode = parts[1]
        label_data[label_id] = {"name": name, "mode": mode}
        for img_name in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, FACE_SIZE)
            faces.append(img)
            labels.append(label_id)
        label_id += 1

    if len(faces) == 0:
        print("No face images found.")
        return False

    recognizer.train(faces, np.array(labels))
    recognizer.save(FACE_MODEL_PATH)
    with open(FACE_LABELS_PATH, "w", encoding="utf-8") as f:
        json.dump(label_data, f, indent=4)

    print("Model trained. Users:", label_data)
    return True


def register_new_face_user(name, mode):
    mode = mode.lower().strip()
    if mode not in ["m", "f"]:
        mode = "m"

    safe_name = name.strip().replace(" ", "_")
    safe_name = "".join(ch for ch in safe_name if ch.isalnum() or ch == "_")
    save_dir = os.path.join(FACE_DIR, safe_name + "_" + mode)
    os.makedirs(save_dir, exist_ok=True)

    face_cascade = get_face_detector()
    cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
    if not cap.isOpened():
        print("Camera could not open.")
        return False

    count = 0
    print("Registering:", safe_name, mode)
    speak("Look at the camera.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=5, minSize=(80, 80))

        for (x, y, w, h) in faces:
            face_img = gray[y:y+h, x:x+w]
            face_img = cv2.resize(face_img, FACE_SIZE)
            cv2.imwrite(os.path.join(save_dir, str(count) + ".jpg"), face_img)
            count += 1
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
            cv2.putText(frame, f"Saving {count}/{SAMPLES_NEEDED}",
                        (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            time.sleep(0.08)
            break

        cv2.imshow("Register New Face", frame)

        if count >= SAMPLES_NEEDED:
            speak("Done.")
            break
        if cv2.waitKey(1) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()

    if count >= 10:
        train_face_model()
        return True

    print("Not enough samples.")
    return False


def auto_face_login_or_register(conn):
    face_cascade = get_face_detector()
    cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
    if not cap.isOpened():
        print("Camera could not open.")
        return False, None, None

    recognizer = None
    label_data = None

    if face_model_exists():
        recognizer, label_data = load_face_model()
        speak("Face login started.")
    else:
        speak("No users registered. Look at the camera.")

    unknown_start_time = None
    last_name = ""
    hit_count = 0

    print("Face login — ESC to skip.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=5, minSize=(80, 80))

        face_found = False
        recognized = False

        for (x, y, w, h) in faces:
            face_found = True
            face_img = cv2.resize(gray[y:y+h, x:x+w], FACE_SIZE)

            if recognizer is not None:
                label_id, confidence = recognizer.predict(face_img)
                label_id_str = str(label_id)

                if label_id_str in label_data and confidence < RECOGNITION_CONFIDENCE_LIMIT:
                    name = label_data[label_id_str]["name"]
                    mode = label_data[label_id_str].get("mode", label_data[label_id_str].get("gender", "m"))

                    recognized = True
                    unknown_start_time = None

                    if name == last_name:
                        hit_count += 1
                    else:
                        last_name = name
                        hit_count = 1

                    cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
                    cv2.putText(frame, f"{name} {hit_count}/{REQUIRED_LOGIN_HITS}",
                                (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

                    if hit_count >= REQUIRED_LOGIN_HITS:
                        create_user_progress(name)
                        level = get_user_progress(name)

                        send_message(conn, f"LOGIN:{name}:{mode}:{level}")
                        send_word_pools(conn)
                        send_level_info(conn, mode, level)

                        cap.release()
                        cv2.destroyAllWindows()
                        speak("Welcome " + name)
                        print(f"Logged in: {name} mode={mode} level={level}")
                        return True, name, mode

                else:
                    cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 255), 2)
                    cv2.putText(frame, "Unknown", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

            else:
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 255), 2)
                cv2.putText(frame, "New face", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

            break

        if face_found and not recognized:
            if unknown_start_time is None:
                unknown_start_time = time.time()
                speak("Unknown face. Hold still.")

            elapsed = time.time() - unknown_start_time
            cv2.putText(frame, f"Unknown: {round(elapsed, 1)}s",
                        (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

            if elapsed >= UNKNOWN_SECONDS_BEFORE_REGISTER:
                cap.release()
                cv2.destroyAllWindows()

                new_name = get_user_name_by_voice()
                new_mode = get_user_mode_by_voice()

                registered = register_new_face_user(new_name, new_mode)

                if registered:
                    create_user_progress(new_name)
                    level = get_user_progress(new_name)
                    send_message(conn, f"LOGIN:{new_name}:{new_mode}:{level}")
                    send_word_pools(conn)
                    send_level_info(conn, new_mode, 1)
                    speak("Welcome " + new_name)
                    return True, new_name, new_mode

                return False, None, None

        elif not face_found:
            unknown_start_time = None
            hit_count = 0

        cv2.imshow("Face Login", frame)

        if cv2.waitKey(1) & 0xFF == 27:
            print("Login skipped.")
            break

    cap.release()
    cv2.destroyAllWindows()
    return False, None, None


# --- gesture templates ($1 recognizer) ---
# extracts the right index fingertip trail from a video
# used to build gesture templates for dollarpy

def getPoints(videoURL, label):
    cap = cv2.VideoCapture(videoURL)
    points = []
    stroke_id = 1

    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            results = holistic.process(image)
            image.flags.writeable = True

            if results.right_hand_landmarks:
                tip = results.right_hand_landmarks.landmark[8]
                h, w, _ = frame.shape
                points.append(Point(int(tip.x * w), int(tip.y * h), stroke_id))

    cap.release()
    cv2.destroyAllWindows()
    return points


templates = []


def add_template(label, video_path):
    points = getPoints(video_path, label)
    print(label, video_path, "points =", len(points))
    if len(points) > 0:
        templates.append(Template(label, points))
    else:
        print("Skipped (no points):", video_path)


add_template("circle",      "Dataset/Dataset/circle/circle_1.mp4")
add_template("circle",      "Dataset/Dataset/circle/circle_2.mp4")
add_template("slide_left",  "Dataset/Dataset/slide_left/slide_left_1.mp4")
add_template("slide_left",  "Dataset/Dataset/slide_left/slide_left_2.mp4")
add_template("slide_right", "Dataset/Dataset/slide_right/slide_right_1.mp4")
add_template("slide_right", "Dataset/Dataset/slide_right/slide_right_2.mp4")
add_template("fist",        "Dataset/Dataset/fist/fist_1.mp4")
add_template("fist",        "Dataset/Dataset/fist/fist_2.mp4")
add_template("peace",       "Dataset/Dataset/peace/peace_1.mp4")
add_template("peace",       "Dataset/Dataset/peace/peace_2.mp4")

print("Templates loaded:", len(templates))

if len(templates) == 0:
    raise Exception("No templates loaded.")

dollar_recognizer = Recognizer(templates)


# --- gesture sending ---

last_label = ""
last_time = 0


def send_gesture(conn, label):
    global last_label, last_time
    now = time.time()

    if label == last_label and now - last_time < GESTURE_COOLDOWN:
        return

    mapping = {
        "slide_right": "GESTURE:RIGHT",
        "slide_left":  "GESTURE:LEFT",
        "circle":      "GESTURE:CIRCLE",
        "fist":        "GESTURE:FIST",
        "peace":       "GESTURE:PEACE",
    }

    if label not in mapping:
        return

    send_message(conn, mapping[label])
    last_label = label
    last_time = now


# --- skeleton zone detection ---
# maps the fingertip position to one of three word slots on screen
# only fires in the lower half of the camera frame to avoid conflicts
# with gestures made at the top
#
# box positions measured from the actual game screenshot (1456px wide):
#   subject: 130-390, verb: 610-860, object: 1080-1340
# these get scaled down to camera coords using the window width

last_zone = ""
last_zone_time = 0

GAME_WINDOW_WIDTH = 1920
IS_LASER_MODE = False


def get_skeleton_zone(x, y, frame_width, frame_height):
    if y < frame_height // 2:
        return None

    scale = GAME_WINDOW_WIDTH / frame_width

    if int(130 / scale) <= x <= int(390 / scale):
        return "SUBJECT"
    elif int(610 / scale) <= x <= int(860 / scale):
        return "VERB"
    elif int(1080 / scale) <= x <= int(1340 / scale):
        return "OBJECT"
    return None


def send_skeleton_zone(conn, zone):
    global last_zone, last_zone_time
    now = time.time()
    if zone == last_zone and now - last_zone_time < SKELETON_ZONE_COOLDOWN:
        return
    send_message(conn, "ZONE:" + zone)
    last_zone = zone
    last_zone_time = now


# --- yolo object detection ---
# ball, book, apple each send a distinct command to C#
# separate from hand gestures

last_object = ""
last_object_time = 0


def send_detected_object(conn, object_name):
    global last_object, last_object_time
    now = time.time()

    if object_name == last_object and now - last_object_time < OBJECT_COOLDOWN:
        return

    mapping = {
        "sports ball": "OBJECT:BALL",
        "book":        "OBJECT:BOOK",
        "apple":       "OBJECT:APPLE",
    }

    if object_name not in mapping:
        return

    send_message(conn, mapping[object_name])
    last_object = object_name
    last_object_time = now


def process_yolo_detection(conn, frame):
    yolo_results = yolo_model(frame, verbose=False)
    for result in yolo_results:
        for box in result.boxes:
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
            object_name = yolo_model.names[class_id]
            if confidence >= YOLO_CONFIDENCE and object_name in ["sports ball", "book", "apple"]:
                print("YOLO:", object_name, round(confidence, 2))
                send_detected_object(conn, object_name)


# --- emotion detection ---
# uses mediapipe face landmarks to estimate expression
# ratios based on mouth width, openness, and corner drop

last_emotion = ""
last_emotion_time = 0


def send_emotion(conn, emotion_name):
    global last_emotion, last_emotion_time
    now = time.time()
    emotion_name = emotion_name.upper().strip()

    if emotion_name == last_emotion and now - last_emotion_time < EMOTION_COOLDOWN:
        return
    if emotion_name not in ["HAPPY", "NEUTRAL", "SAD", "SURPRISE"]:
        return

    send_message(conn, "EMOTION:" + emotion_name)
    last_emotion = emotion_name
    last_emotion_time = now


def classify_expression_from_face_landmarks(face_landmarks):
    if face_landmarks is None:
        return "NEUTRAL"

    lm = face_landmarks.landmark
    mouth_width  = abs(lm[291].x - lm[61].x)
    mouth_open   = abs(lm[14].y  - lm[13].y)
    face_height  = abs(lm[152].y - lm[1].y)

    if face_height == 0:
        return "NEUTRAL"

    open_ratio  = mouth_open / face_height
    smile_ratio = mouth_width / face_height
    sad_ratio   = ((lm[61].y + lm[291].y) / 2) - ((lm[37].y + lm[267].y) / 2)

    print("mouth_open:", round(open_ratio, 2), "smile:", round(smile_ratio, 2), "sad:", round(sad_ratio, 3))

    if open_ratio  > 0.22: return "SURPRISE"
    if smile_ratio > 0.95: return "HAPPY"
    if sad_ratio   > 0.035: return "SAD"
    return "NEUTRAL"


# --- main camera loop ---

def recognize_live_gesture(conn, current_user, current_mode="m"):
    cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
    if not cap.isOpened():
        print("Camera could not open.")
        return

    conn.setblocking(False)

    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        points = []
        stroke_id = 1
        tracking = False
        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            h, w, _ = frame.shape

            try:
                data = conn.recv(1024)
                if data:
                    incoming = data.decode("utf-8").strip()
                    print("From C#:", incoming)

                    if incoming.startswith("WINDOWSIZE:"):
                        try:
                            global GAME_WINDOW_WIDTH
                            GAME_WINDOW_WIDTH = int(incoming.split(":")[1])
                        except Exception as e:
                            print("WINDOWSIZE error:", e)

                    elif incoming.startswith("MODE:"):
                        global IS_LASER_MODE
                        IS_LASER_MODE = (incoming.split(":")[1].strip() == "LASER")
                        print("Mode:", "LASER" if IS_LASER_MODE else "GESTURE")

                    elif incoming.startswith("LEVELUP:") and current_user:
                        try:
                            new_level = int(incoming.split(":")[1])
                            update_user_progress(current_user, new_level)
                            speak("Level " + str(new_level) + " unlocked!")
                            send_level_info(conn, current_mode, new_level)
                        except Exception as e:
                            print("LEVELUP error:", e)

                    elif incoming == "WIN" and current_user:
                        speak("Congratulations! You win!")
                        print("All levels complete:", current_user)

            except BlockingIOError:
                pass
            except Exception as e:
                print("Receive error:", e)

            if frame_count % YOLO_EVERY_N_FRAMES == 0:
                process_yolo_detection(conn, frame)

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            results = holistic.process(image)
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            if frame_count % EMOTION_EVERY_N_FRAMES == 0:
                emotion = classify_expression_from_face_landmarks(results.face_landmarks)
                if emotion != "NEUTRAL":
                    print(f"[EMOTION] {emotion} frame {frame_count}")
                send_emotion(conn, emotion)

            if results.right_hand_landmarks:
                mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

                tip = results.right_hand_landmarks.landmark[8]
                x = int(tip.x * w)
                y = int(tip.y * h)

                points.append(Point(x, y, stroke_id))
                cv2.circle(image, (x, y), 8, (0, 255, 0), -1)
                tracking = True

                zone = get_skeleton_zone(x, y, w, h)
                if zone:
                    send_skeleton_zone(conn, zone)

                if IS_LASER_MODE:
                    send_message(conn, f"FINGER:{x},{y}")

            else:
                if tracking and len(points) > 10:
                    try:
                        result = dollar_recognizer.recognize(points)
                        if result is not None:
                            print("Predicted:", result[0], "Score:", result[1])
                            send_gesture(conn, result[0])
                    except Exception as e:
                        print("Recognition error:", str(e))

                points = []
                tracking = False

            cv2.imshow("HCI Game — Camera Feed", image)

            if cv2.waitKey(1) & 0xFF == 27:
                break

    cap.release()
    cv2.destroyAllWindows()


# --- entry point ---

async def main():
    # change to "2" to open the teacher panel instead
    choice = "1"

    if choice == "2":
        teacher_panel()
        print("Done. Change choice back to 1 to start the game.")
        return

    soc = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    soc.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    soc.bind((SOCKET_HOST, SOCKET_PORT))
    soc.listen(1)

    print("Waiting for C# to connect on port 5000...")

    conn, addr = soc.accept()
    print("C# connected:", addr)

    logged_in, current_user, current_mode = auto_face_login_or_register(conn)

    if not logged_in:
        send_message(conn, "LOGIN:Guest:m:1")
        send_word_pools(conn)
        send_level_info(conn, "m", 1)
        current_user = "Guest"
        current_mode = "m"

    recognize_live_gesture(conn, current_user, current_mode)

    conn.close()
    soc.close()


await main()

circle Dataset/Dataset/circle/circle_1.mp4 points = 69
circle Dataset/Dataset/circle/circle_2.mp4 points = 105
slide_left Dataset/Dataset/slide_left/slide_left_1.mp4 points = 40
slide_left Dataset/Dataset/slide_left/slide_left_2.mp4 points = 26
slide_right Dataset/Dataset/slide_right/slide_right_1.mp4 points = 12
slide_right Dataset/Dataset/slide_right/slide_right_2.mp4 points = 15
fist Dataset/Dataset/fist/fist_1.mp4 points = 59
fist Dataset/Dataset/fist/fist_2.mp4 points = 51
peace Dataset/Dataset/peace/peace_1.mp4 points = 64
peace Dataset/Dataset/peace/peace_2.mp4 points = 77
Templates loaded: 10
Waiting for C# to connect on port 5000...
C# connected: ('127.0.0.1', 64223)
Face login — ESC to skip.
Yusuf_use already saved at level 2
Sent: LOGIN:Yusuf_use:m:2
Sent: WORDS:subjects:I,You,He,She
Sent: WORDS:verbs:eat,read,kick,drink
Sent: WORDS:objects:apple,book,ball,milk
Sent: TOTALLEVELS:4
Sent: LEVELANSWER:He kick ball
Logged in: Yusuf_use mode=m level=2
From C#: WINDOWSIZE:1540
mou

In [1]:
print("Test started")

import cv2
print("cv2 OK")

import mediapipe as mp
print("mediapipe OK", mp.__version__, hasattr(mp, "solutions"))

from ultralytics import YOLO
print("ultralytics import OK")

model = YOLO("yolov8n.pt")
print("YOLO model loaded")

print("All good")

Test started
cv2 OK
mediapipe OK 0.10.9 True
ultralytics import OK
YOLO model loaded
All good


In [6]:
import cv2

for i in range(5):
    cap = cv2.VideoCapture(i, cv2.CAP_DSHOW)
    ret, frame = cap.read()

    if ret:
        print("Camera found at index:", i)
        cv2.imshow(f"Camera {i}", frame)
        cv2.waitKey(1000)
        cv2.destroyAllWindows()
    else:
        print("No camera at index:", i)

    cap.release()

No camera at index: 0
Camera found at index: 1
No camera at index: 2
No camera at index: 3
No camera at index: 4


In [ ]:
# This is used to close the socket connection after finishing the communication
soc.close()

In [2]:
!pip install bleak


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
